# Prep: BigBird fine-tuning for long news articles

Fine-tunes `ilos-vigil/bigbird-small-indonesian-nli` (max length 512) with the Hugging Face `Trainer` and weighted F1.

In [ ]:
DATA_DIR = "../data"

In [2]:
import os

os.environ['CUDA_VISIBLE_DEVICES'] = '0'

## Import Library

In [3]:
!pip install -q evaluate datasets transformers

In [4]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu115

In [6]:
import pandas as pd
import numpy as np

from transformers import AutoTokenizer, TFAutoModelForSequenceClassification, TFBertModel, TFAutoModel, \
    TrainingArguments, Trainer, AutoModelForSequenceClassification, EarlyStoppingCallback, BigBirdModel
import evaluate, torch
import tensorflow as tf
import keras.backend as K
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.initializers import TruncatedNormal
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.metrics import CategoricalAccuracy
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.layers import Input, Dense

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, recall_score, precision_score

In [7]:
df = pd.read_csv(f"{DATA_DIR}/gemastik_22.csv")
df.head()

,id,sumber,kodekat,kodesubkat,kategori,subkategori,katakunci,tanggal,judul_berita,konten_berita,nama_tokoh,jabatan,organisasi,lokasi,alias,kutipan,sentimen_kutipan,sentimen_berita
0,568576e9e95,Detik,Q,Q,Jasa Kesehatan dan Kegiatan Sosial Swasta,Jasa Kesehatan dan Kegiatan Sosial Swasta,Jumlah pasien,2021-06-17,"BOR Sudah 89 Persen, RSUD Soreang Bandung Siap...",Bandung -\n\nSatuan Tugas Penanganan COVID-19 ...,[],['Bupati Bandung'],"['Bed Occupancy Ratio (BOR)', 'Pemda', 'BOR']","['Bandung', 'Kabupaten Bandung', 'Kota Bandung']",[],"['mengatakan, telah mengadakan rapat dan memut...",Netral,Netral
1,80f931fe95e,Detik,D,D2,Pengadaan Listrik dan Gas,Pengadaan Gas dan Produksi Es,Jaringan Gas,2021-03-18,"Honda CB650R Indonesia Dapat Penyegaran, Harga...",Jakarta -\n\nAstra Honda Motor (AHM) resmi men...,"['Biasa Mejeng', 'Resmi Dijual']",[],"['Big', 'LED']","['Jakarta', 'Indonesia']",[],"['""Rasa bangga itu menyempurnakan gaya hidup m...",Positif,Positif
2,1bd2af6ef23,Antara,C,C8,Industri Pengolahan,"Industri Kimia, Farmasi dan Obat Tradisional",farmasi,2021-02-23,Vaksin COVID-19 Pfizer disetujui satu dari tig...,Seoul (ANTARA) - Panel ahli pertama dari tiga ...,['Mulyo Sunyoto'],[],"['Universitas Oxford', 'BioNTech']","['Korea Selatan', 'Amerika Serikat', 'Korsel',...",['¬©'],[],Netral,Positif
3,ac2b55af24c,Detik,A,A2,"Pertanian, Kehutanan, dan Perikanan",Kehutanan dan Penebangan Kayu,bambu,2021-01-26,"TPU Srengseng Sawah Penuh, DKI Rujuk Jenazah C...","Jakarta - TPU Srengseng Sawah, Jakarta Selatan...",['Ivan Nurcahyo'],[],"['Pemprov DKI', 'Dinas Pertamanan']","['Jakarta', 'Jakarta Selatan', 'DKI Jakarta', ...",[],"['""Kita sebutnya TPU Bambu Wulung, sebenarnya ...",Positif,Positif
4,3f50747a1d4,Antara,Q,Q,Jasa Kesehatan dan Kegiatan Sosial Swasta,Jasa Kesehatan dan Kegiatan Sosial Swasta,wabah penyakit,2021-05-04,Media Korea Utara sebut vaksin COVID-19 bukan ...,Seoul (ANTARA) - Media pemerintah Korea Utara ...,['Edwin Salvador'],[],['Partai Buruh'],"['Korea Utara', 'Korea Selatan', 'China', 'Kor...",['¬©'],['mengatakan wabah di sana tidak dapat dikesam...,Netral,Netral


In [8]:
df['judul_konten'] = df['judul_berita'] + ' ' + df['konten_berita']
df['judul_konten'].head()

0    BOR Sudah 89 Persen, RSUD Soreang Bandung Siap...
1    Honda CB650R Indonesia Dapat Penyegaran, Harga...
2    Vaksin COVID-19 Pfizer disetujui satu dari tig...
3    TPU Srengseng Sawah Penuh, DKI Rujuk Jenazah C...
4    Media Korea Utara sebut vaksin COVID-19 bukan ...
Name: judul_konten, dtype: object

In [9]:
# MODEL = 'cardiffnlp/twitter-roberta-base-sentiment-latest'
# MODEL = 'ayameRushia/bert-base-indonesian-1.5G-sentiment-analysis-smsa'
MODEL = 'ilos-vigil/bigbird-small-indonesian-nli'

In [10]:
df['judul_konten'].tolist()[:5]

['BOR Sudah 89 Persen, RSUD Soreang Bandung Siapkan 100 Kasur Bandung -\n\nSatuan Tugas Penanganan COVID-19 Kabupaten Bandung memutuskan akan menggunakan RSUD Soreang menjadi lokasi penanganan pasien COVID-19. Sebanyak 100 kasur akan disiapkan di RSUD Soreang.\n\nSebelumnya, Kabupaten Bandung masuk dalam zona risiko tinggi atau zona merah. Per tanggal 16 Juni 2021, Bed Occupancy Ratio (BOR) atau keterisian ruang isolasi mencapai 89 persen.\n\nHal tersebut diungkapkan oleh Bupati Bandung Dadang Supriatna. Ia mengatakan, telah mengadakan rapat dan memutuskan untuk menggunakan RSUD Soreang dijadikan rumah sakit rujukan bagi pasien COVID-19.\n\n"Kita sudah instruksikan dan rapat evaluasi untuk bed, karena RSUD Soreang lama sudah ada pemindahan dan ijin sudah hampir selesai. Maka kami sudah sepakat akan menambah 100 bed isolasi dalam membantu masyarakat," ujar Bupati Bandung Dadang Supriatna, di Komplek Pemda Bandung, Kamis (1762021).\n\nSelain di RSUD Soreang, sejumlah rumah sakit rujukan 

In [11]:
LABEL_MAP = {
    "Negatif": 0,
    "Netral": 1,
    "Positif": 2,
}

INV_LABEL_MAP = {v: k for k, v in LABEL_MAP.items()}

X = df['judul_konten'].values
y = np.array(df["sentimen_berita"].map(LABEL_MAP))
print(X.shape, y.shape)
y[:10]

(5289,) (5289,)


array([1, 2, 2, 2, 1, 1, 2, 1, 2, 2])

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y)

X_train.shape, X_test.shape, y_train.shape, y_test.shape

((4231,), (1058,), (4231,), (1058,))

In [13]:
tokenizer = AutoTokenizer.from_pretrained(MODEL)

In [14]:
MAX_LENGTH = 512

tokenized_text_detail_train = tokenizer(
    text=X_train.tolist(),
    add_special_tokens=True,
    max_length=MAX_LENGTH,
    truncation=True,
    padding=True,
    return_tensors='pt',
    return_token_type_ids=False,
    return_attention_mask=True,
    verbose=True
)

tokenized_text_detail_test = tokenizer(
    text=X_test.tolist(),
    add_special_tokens=True,
    max_length=MAX_LENGTH,
    truncation=True,
    padding=True,
    return_tensors='pt',
    return_token_type_ids=False,
    return_attention_mask=True,
    verbose=True
)

tokenized_text_detail_train

{'input_ids': tensor([[    3,   605,  4518,  ..., 12272,   173,     4],
        [    3,   180,  3871,  ...,   751,   692,     4],
        [    3,  1951,  1321,  ..., 18160,   619,     4],
        ...,
        [    3,  1345,   147,  ...,     0,     0,     0],
        [    3,  8539,    85,  ...,   500,  1951,     4],
        [    3, 15161,  1292,  ...,  1525,   144,     4]]), 'attention_mask': tensor([[1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 1, 1, 1],
        [1, 1, 1,  ..., 1, 1, 1]])}

In [15]:
token_ids_train = np.array(tokenized_text_detail_train["input_ids"])
token_masks_train = np.array(tokenized_text_detail_train["attention_mask"])
token_ids_test = np.array(tokenized_text_detail_test["input_ids"])
token_masks_test = np.array(tokenized_text_detail_test["attention_mask"])

token_ids_train.shape, token_masks_train.shape, token_ids_test.shape, token_masks_test.shape

((4231, 512), (4231, 512), (1058, 512), (1058, 512))

## Using Trainer

In [16]:
# Create torch dataset
class Dataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels=None):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        if self.labels:
            item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.encodings["input_ids"])

In [17]:
train_dataset = Dataset({
    "input_ids": token_ids_train,
    "attention_mask": token_masks_train,
}, y_train.tolist())
test_dataset = Dataset({
    "input_ids": token_ids_test,
    "attention_mask": token_masks_test,
}, y_test.tolist())

In [18]:
bigbird_model = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=3)

In [19]:
args = TrainingArguments(
    output_dir="test_trainer",
    evaluation_strategy="steps",
    eval_steps=500,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=10,
    seed=42,
    load_best_model_at_end=True,
)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return f1_metric.compute(predictions=predictions, references=labels, average="weighted")

f1_metric = evaluate.load("f1", average="weighted")


trainer = Trainer(
    model=bigbird_model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

In [20]:
trainer.train()

Attention type 'block_sparse' is not possible if sequence_length: 512 <= num global tokens: 2 * config.block_size + min. num sliding tokens: 3 * config.block_size + config.num_random_blocks * config.block_size + additional buffer: config.num_random_blocks * config.block_size = 704 with config.block_size = 64, config.num_random_blocks = 3. Changing attention type to 'original_full'...


Step,Training Loss,Validation Loss


TrainOutput(global_step=1330, training_loss=0.4209721873577376, metrics={'train_runtime': 128.1357, 'train_samples_per_second': 330.197, 'train_steps_per_second': 10.38, 'total_flos': 1707552336107520.0, 'train_loss': 0.4209721873577376, 'epoch': 10.0})

In [21]:
trainer.evaluate()

{'eval_loss': 0.9220414757728577,
 'eval_f1': 0.6469084468616438,
 'eval_runtime': 0.7996,
 'eval_samples_per_second': 1323.136,
 'eval_steps_per_second': 42.52,
 'epoch': 10.0}

In [22]:
torch.cuda.empty_cache()